# 🛡️ Vendor Payment Guard
### A local AI agent that checks every vendor invoice *before* money leaves the company

**Course:** Agentic AI (STAI343) · **Built with:** LangGraph (state machine) + Ollama (local LLM, optional)

---

**The problem.** Fraudsters hack or copy a real supplier's email and send an invoice that looks genuine,
except the **bank account has been changed** to theirs. Indian companies have lost ₹2 crore (Chennai) and
₹2.35 crore (Pune) this way. Small accounts teams check invoices by hand, under time pressure.

**Our solution.** A security checkpoint for payments. Every invoice passes through a set of guards:

| Result | When |
|---|---|
| ✅ **APPROVE** | everything matches our records |
| 📞 **CALL THE VENDOR** | something is suspicious, so we phone the vendor on the number *we* have on file |
| ⛔ **BLOCK** | clearly dangerous, or the vendor denies sending it |
| ❌ **REJECT** | the invoice is incomplete and can't even be checked |

**Golden rule:** *the LLM reads, the rules decide.* The local LLM may help read a messy invoice,
but approve/block decisions are always made by fixed, auditable rules.

**Notebook map**
1. Settings (our finance policy)
2. Mock company data
3. Incoming test invoices
4. Tools: small functions the agent calls
5. State: the agent's memory
6. Nodes
7. Build the graph
8. Run the agent
9. *(next steps)* Parallel checks → Routing → Loop → Visuals → Local LLM

## 1. Settings: our "finance policy"

Everything the agent decides is based on these numbers. Change one, re-run, and the agent behaves differently.

- **`RISK_POINTS`**: each warning flag adds points to an invoice's risk score.
- **`APPROVE_BELOW` / `BLOCK_FROM`**: the score decides the route: approve, call the vendor, or block.
- **`MAX_CALLS`**: the **loop guard**. We phone a vendor at most this many times, then stop.

In [ ]:
# ---- Settings: the finance policy of our agent ----

USE_LLM   = True                  # True  = local LLM (Ollama) reads the invoice text
LLM_MODEL = "qwen3:4b-instruct"   # False = simple Python parser reads it

MAX_CALLS = 3                     # loop guard: phone the vendor at most 3 times

RISK_POINTS = {                   # how dangerous each warning flag is
    "DUPLICATE_INVOICE":    100,
    "BANK_DETAILS_CHANGED":  50,
    "NO_MATCHING_PO":        40,
    "AMOUNT_EXCEEDS_PO":     30,
    "GOODS_NOT_RECEIVED":    30,
    "INVALID_GSTIN":         40,
    "GSTIN_MISMATCH":        40,
}

APPROVE_BELOW = 30                # score  0-29 -> approve
BLOCK_FROM    = 80                # score 80+   -> block      (30-79 -> call the vendor)

## 2. Mock company data

Our pretend company is **Shakti Auto Components, Pune**. It buys raw material from 5 vendors.
In a real company this data lives in an accounting system (Tally, Zoho Books, SAP). Here it's plain Python dictionaries.

| Table | What it holds | Why the agent needs it |
|---|---|---|
| `VENDORS` | trusted vendor master: GSTIN, bank account, **phone on file** | to spot changed bank details and to know whom to call |
| `PURCHASE_ORDERS` | what we ordered and the agreed amount | an invoice must match a real order |
| `GOODS_RECEIPTS` | proof the goods arrived (GRN) | never pay for goods we didn't get |
| `PAID_INVOICES` | invoices already paid | to catch duplicates |
| `CALLBACK_SCRIPT` | what each vendor says when we phone them | simulated phone calls, so every run is the same |

> 🔑 **Key anti-fraud idea:** we always phone the number in **our** vendor master, never a number printed on the invoice.
> The fraudster controls the invoice. They don't control our records.

In [ ]:
# ---- Trusted vendor master (our own records) ----
VENDORS = {
    "V001": {"name": "Sharma Steel Traders", "gstin": "27AABCS1234F1Z5",
             "bank_account": "50100234567", "phone": "+91-90000-10001"},
    "V002": {"name": "Patil Plastics",       "gstin": "27AACCP5678K1Z2",
             "bank_account": "31200456789", "phone": "+91-90000-10002"},
    "V003": {"name": "Gupta Bearings",       "gstin": "27AADFG9012L1Z8",
             "bank_account": "00912345678", "phone": "+91-90000-10003"},
    "V004": {"name": "Rao Rubber Works",     "gstin": "29AAEFR3456M1Z1",
             "bank_account": "20456789012", "phone": "+91-90000-10004"},
    "V005": {"name": "Mehta Packaging",      "gstin": "27AAGFM7890N1Z4",
             "bank_account": "60011223344", "phone": "+91-90000-10005"},
}

# ---- Purchase orders: what we ordered, and for how much (INR) ----
PURCHASE_ORDERS = {
    "PO-1001": {"vendor_id": "V001", "amount": 450000},
    "PO-1002": {"vendor_id": "V002", "amount": 120000},
    "PO-1003": {"vendor_id": "V003", "amount": 275000},
    "PO-1004": {"vendor_id": "V004", "amount":  98000},
    "PO-1005": {"vendor_id": "V005", "amount":  60000},
}

# ---- Goods receipts: which POs have actually been delivered ----
GOODS_RECEIPTS = {
    "PO-1001": "GRN-5001",
    "PO-1002": "GRN-5002",
    "PO-1003": "GRN-5003",
    "PO-1004": "GRN-5004",
    "PO-1005": "GRN-5005",
}

# ---- Invoices we have already paid: (vendor_id, invoice_no) ----
PAID_INVOICES = {
    ("V001", "SST/2026/0398"),
    ("V002", "PP/26/117"),
}

# ---- Simulated phone calls: what the vendor says on call 1, call 2, ... ----
# In real life a person phones the vendor. Here the answers are scripted
# so every run gives the same result (a deterministic demo).
CALLBACK_SCRIPT = {
    "+91-90000-10002": ["no_answer", "confirmed"],   # Patil: picks up on the 2nd call, confirms
    "+91-90000-10003": ["denied"],                   # Gupta: "We never changed our bank!" -> fraud
    "+91-90000-10004": [],                           # Rao: never picks up
}

print(f"Loaded {len(VENDORS)} vendors, {len(PURCHASE_ORDERS)} purchase orders, "
      f"{len(GOODS_RECEIPTS)} goods receipts, {len(PAID_INVOICES)} paid invoices.")

## 3. Incoming test invoices

These 7 invoices are our test cases. Each one tests a different path through the agent.

| # | Invoice | What's special | Expected result |
|---|---|---|---|
| 1 | Sharma Steel | everything matches | ✅ APPROVE |
| 2 | Sharma Steel | already paid before (duplicate) | ⛔ BLOCK |
| 3 | Patil Plastics | new bank account, vendor confirms on the 2nd call | ✅ APPROVE after the loop |
| 4 | Gupta Bearings | new bank account, vendor says *"we never sent this"* | ⛔ BLOCK: **fraud caught** |
| 5 | Rao Rubber Works | new bank account, vendor never answers | ⛔ BLOCK after 3 calls (loop guard) |
| 6 | "Mehta Packaging" | fake GSTIN and a new bank account | ⛔ BLOCK |
| 7 | Mehta Packaging | PO number and amount missing | ❌ REJECT |

In [ ]:
TEST_INVOICES = {
"1. Clean invoice": '''
TAX INVOICE
Invoice No: SST/2026/0412
Vendor: Sharma Steel Traders
PO Number: PO-1001
Amount: 450000
GSTIN: 27AABCS1234F1Z5
Bank Account: 50100234567
''',

"2. Duplicate invoice": '''
TAX INVOICE
Invoice No: SST/2026/0398
Vendor: Sharma Steel Traders
PO Number: PO-1001
Amount: 450000
GSTIN: 27AABCS1234F1Z5
Bank Account: 50100234567
''',

"3. Bank changed, vendor confirms": '''
TAX INVOICE
Invoice No: PP/26/131
Vendor: Patil Plastics
PO Number: PO-1002
Amount: 120000
GSTIN: 27AACCP5678K1Z2
Bank Account: 77889900112
''',

"4. Bank changed, vendor denies (fraud)": '''
TAX INVOICE
Invoice No: GB-2026-0077
Vendor: Gupta Bearings
PO Number: PO-1003
Amount: 275000
GSTIN: 27AADFG9012L1Z8
Bank Account: 66554433221
''',

"5. Bank changed, vendor never answers": '''
TAX INVOICE
Invoice No: RRW/0931
Vendor: Rao Rubber Works
PO Number: PO-1004
Amount: 98000
GSTIN: 29AAEFR3456M1Z1
Bank Account: 11223344556
''',

"6. Fake GSTIN + bank changed": '''
TAX INVOICE
Invoice No: MP/2026/552
Vendor: Mehta Packaging
PO Number: PO-1005
Amount: 60000
GSTIN: 27AAGFM78901Z4
Bank Account: 99887766554
''',

"7. Incomplete invoice": '''
TAX INVOICE
Invoice No: MP/2026/553
Vendor: Mehta Packaging
GSTIN: 27AAGFM7890N1Z4
Bank Account: 60011223344
''',
}

for name in TEST_INVOICES:
    print(name)

## 4. Tools: small functions the agent calls

A **tool** is a small function that does **one** job, usually *looking something up*.
The agent's nodes (next step) call these tools to make decisions. This is **function calling**:
the agent doesn't know the answers itself, so it calls the right function to find out.

| Tool | Question it answers |
|---|---|
| `find_vendor(name)` | Is this vendor in our master? What are their real details? |
| `find_purchase_order(po_number)` | Did we really order this? For how much? |
| `find_goods_receipt(po_number)` | Did the goods actually arrive? |
| `is_already_paid(vendor_id, invoice_no)` | Have we paid this invoice before? |
| `is_valid_gstin_format(gstin)` | Is the GSTIN a valid 15-character Indian GST number? |
| `call_vendor(phone, attempt)` | *(simulated)* What did the vendor say on this call? |

In [ ]:
import re

def find_vendor(name):
    # Look up a vendor in our trusted master by name. Returns their record, or None if unknown.
    for vendor_id, vendor in VENDORS.items():
        if vendor["name"].lower() == name.strip().lower():
            return {"vendor_id": vendor_id, **vendor}
    return None

def find_purchase_order(po_number):
    # Return the purchase order, or None if we never placed it.
    return PURCHASE_ORDERS.get(po_number)

def find_goods_receipt(po_number):
    # Return the goods receipt number, or None if the goods never arrived.
    return GOODS_RECEIPTS.get(po_number)

def is_already_paid(vendor_id, invoice_no):
    # True if we have already paid this exact invoice.
    return (vendor_id, invoice_no) in PAID_INVOICES

# GSTIN = 2-digit state code + 10-char PAN + entity number + 'Z' + check character
GSTIN_PATTERN = r"[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z][1-9A-Z]Z[0-9A-Z]"

def is_valid_gstin_format(gstin):
    # True if the GSTIN has the correct 15-character format.
    return re.fullmatch(GSTIN_PATTERN, gstin) is not None

def call_vendor(phone, attempt):
    # SIMULATED phone call. Returns "confirmed", "denied" or "no_answer".
    # 'attempt' is 1 for the first call, 2 for the second, and so on.
    answers = CALLBACK_SCRIPT.get(phone, [])
    if attempt <= len(answers):
        return answers[attempt - 1]
    return "no_answer"

### Quick test: do the tools work on their own?

Before building the agent, we test each tool by hand. If a tool is wrong, the agent will be wrong too.

In [ ]:
print("find_vendor('Patil Plastics')       ->", find_vendor("Patil Plastics"))
print("find_vendor('Fake Traders Ltd')     ->", find_vendor("Fake Traders Ltd"))
print("find_purchase_order('PO-1003')      ->", find_purchase_order("PO-1003"))
print("find_purchase_order('PO-9999')      ->", find_purchase_order("PO-9999"))
print("find_goods_receipt('PO-1001')       ->", find_goods_receipt("PO-1001"))
print("is_already_paid('V001', 'SST/2026/0398') ->", is_already_paid("V001", "SST/2026/0398"))
print("is_already_paid('V001', 'SST/2026/0412') ->", is_already_paid("V001", "SST/2026/0412"))
print("is_valid_gstin_format('27AABCS1234F1Z5') ->", is_valid_gstin_format("27AABCS1234F1Z5"))
print("is_valid_gstin_format('27AAGFM78901Z4')  ->", is_valid_gstin_format("27AAGFM78901Z4"))

patil = VENDORS["V002"]["phone"]
rao   = VENDORS["V004"]["phone"]
print("call_vendor(Patil, attempt 1) ->", call_vendor(patil, 1))
print("call_vendor(Patil, attempt 2) ->", call_vendor(patil, 2))
print("call_vendor(Rao,   attempt 5) ->", call_vendor(rao, 5))

## 5. The State: the agent's memory

A normal chatbot is **stateless**: every message starts from zero.
Our agent is **stateful**: one dictionary called the **State** travels through every step,
and each step can read it and add to it.

Think of it as a **file folder that moves from desk to desk** in an accounts office.
Each desk (a **node**) reads the folder, does its one job, and adds a note.

Three words you must know:

| Word | Meaning | In our project |
|---|---|---|
| **State** | the shared folder | `PaymentState` below |
| **Delta** | the small change one node returns (*only* the keys it changed) | e.g. `{"errors": [], "audit_log": ["validate: ok"]}` |
| **Reducer** | the rule for *merging* a delta into the State | no reducer = **replace** the old value · `operator.add` = **append** to the old list |

Look at `risk_flags` and `audit_log` below: they use `Annotated[list, operator.add]`.
That means *"when a node returns this key, append it to the existing list instead of replacing it."*
We'll see why that matters in Step 3, when 4 checks write `risk_flags` **at the same time**.

In [ ]:
import operator
from typing import TypedDict, Annotated

class PaymentState(TypedDict):
    raw_text: str          # the invoice text as received (the input)
    invoice: dict          # fields read from the text: invoice_no, vendor, po_number, amount, gstin, bank_account
    vendor: dict           # OUR trusted record of this vendor (from the vendor master)
    errors: list[str]      # reasons the invoice can't be checked at all (missing fields, unknown vendor)

    risk_flags: Annotated[list[str], operator.add]   # warnings from the checks      -> REDUCER: append
    risk_score: int        # total points of all risk flags
    call_attempts: int     # how many times we phoned the vendor (the LOOP COUNTER)
    vendor_response: str   # what the vendor said on the last call
    decision: str          # final answer: APPROVED / BLOCKED / REJECTED

    audit_log: Annotated[list[str], operator.add]    # one line per node, a full trail  -> REDUCER: append

## 6. Nodes: the desks the folder visits

A **node** is just a Python function:

```
def my_node(state):          # 1. receives the whole State
    ...do one job...
    return {"some_key": value}   # 2. returns ONLY what it changed (the delta)
```

LangGraph takes that delta and merges it into the State using the reducers.

### Node 1 · `read_invoice`: turn text into fields
For now we use a **simple parser** that reads lines like `Vendor: Patil Plastics`.
(In Step 7 we add the local LLM as a second way to read invoices, controlled by `USE_LLM`.)

In [ ]:
# Which label on the invoice maps to which field in our dictionary
LABELS = {
    "Invoice No":   "invoice_no",
    "Vendor":       "vendor",
    "PO Number":    "po_number",
    "Amount":       "amount",
    "GSTIN":        "gstin",
    "Bank Account": "bank_account",
}
REQUIRED_FIELDS = list(LABELS.values())

def parse_invoice_text(text):
    # Simple parser: reads "Label: value" lines from our invoice template.
    invoice = {}
    for line in text.splitlines():
        if ":" in line:
            label, value = line.split(":", 1)
            field = LABELS.get(label.strip())
            if field:
                invoice[field] = value.strip()
    if "amount" in invoice:                               # "4,50,000" -> 450000
        digits = invoice["amount"].replace(",", "")
        invoice["amount"] = int(digits) if digits.isdigit() else None
    return invoice


def read_invoice(state):
    # NODE: read the raw text and pull out the invoice fields.
    invoice = parse_invoice_text(state["raw_text"])
    return {
        "invoice": invoice,
        "audit_log": [f"read_invoice: found {len(invoice)} of {len(REQUIRED_FIELDS)} fields"],
    }

### Node 2 · `validate`: can we even check this invoice?

Before any fraud check, the invoice must be **complete** and from a vendor **we know**.
If not, there's nothing to compare against, so later (Step 4) such invoices go straight to **REJECT**.

In [ ]:
def validate(state):
    # NODE: make sure every required field is present and the vendor is in our master.
    invoice = state["invoice"]
    errors = [f"missing {field}" for field in REQUIRED_FIELDS if not invoice.get(field)]

    vendor = find_vendor(invoice.get("vendor", ""))          # TOOL CALL
    if invoice.get("vendor") and vendor is None:
        errors.append("vendor not in our master")

    note = "ok" if not errors else "; ".join(errors)
    return {
        "vendor": vendor or {},
        "errors": errors,
        "audit_log": [f"validate: {note}"],
    }

## 7. Build the graph

Building a LangGraph agent always has the same 4 moves:

1. `StateGraph(PaymentState)`: create a graph that carries our State
2. `add_node(name, function)`: register each desk
3. `add_edge(a, b)`: draw the arrows (`START` = entry, `END` = exit)
4. `compile()`: turn the blueprint into a runnable agent

**Version 1 is a straight line:** `START → read_invoice → validate → END`.
We'll grow this same cell step by step.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(PaymentState)

builder.add_node("read_invoice", read_invoice)
builder.add_node("validate", validate)

builder.add_edge(START, "read_invoice")
builder.add_edge("read_invoice", "validate")
builder.add_edge("validate", END)

graph = builder.compile()

print(graph.get_graph().draw_ascii())    # drawn locally, no internet needed

## 8. Run the agent

### Watch the deltas for one invoice
`graph.stream(..., stream_mode="updates")` shows **what each node returned** (its delta), one step at a time.
This is the core skill for **tracing the delta**.

In [ ]:
first_invoice = TEST_INVOICES["1. Clean invoice"]

for step, update in enumerate(graph.stream({"raw_text": first_invoice}, stream_mode="updates"), start=1):
    for node_name, delta in update.items():
        print(f"Step {step} · node '{node_name}' returned this delta:")
        for key, value in delta.items():
            print(f"      {key}: {value}")
        print()

### Run all 7 invoices
`graph.invoke(...)` runs the whole graph and returns the **final State**.
Right now the agent can only read and validate, so only invoice 7 should report errors.

In [ ]:
for name, text in TEST_INVOICES.items():
    final_state = graph.invoke({"raw_text": text})
    errors = final_state["errors"] or "none"
    print(f"{name:42} | vendor id: {final_state['vendor'].get('vendor_id', '-'):5} | errors: {errors}")

---
✅ **Step 2 done:** the State is defined, and two nodes run in a straight line through LangGraph.

**Next (Step 3):** add the 4 fraud checks running **in parallel**, all writing to `risk_flags`,
see the **reducer** merge their deltas, and add the `risk_scorer` node.